# Packages

In [1]:
import os
import glob
import json
import sys
import time

import pandas as pd
import numpy as np
import rasterio
import rasterstats
import rioxarray as rxr
import geopandas as gpd
import geemap
from osgeo import gdal

In [2]:
# from google.colab import drive
# drive.mount("/content/gdrive", force_remount=True)
# dir = '/content/gdrive/MyDrive/Work/ADB/SAR/'

dir = 'G:/My Drive/Work/ADB/SAR/'


In [3]:
import ee
# Trigger the authentication flow.
try:
    ee.Initialize(project='ee-sonle96')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-sonle96')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


In [4]:
sys.path.append('G:/My Drive/Work/ADB/SAR/src/gee_s1_ard/python-api')
import border_noise_correction as bnc
import speckle_filter as sf
import terrain_flattening as trf
from src.dhelper import get_month_range, get_last_day_of_month_from_Ym, get_prev_ndays

# Radar Vegetation Index

## Setting Params

In [5]:
params = {'START_DATE': '2022-01-01',
            'STOP_DATE': '2022-02-01',
            'POLARIZATION': 'VVVH',
            'ORBIT' : 'DESCENDING',
            'PLATFORM_NUMBER' : 'A',
            'ORBIT_NUM': None,
            'ROI': ee.Geometry.Polygon([[[-254.1842336, 9.739978],
                                         [-254.1569875, 9.739608],
                                         [-254.1567476, 9.713047],
                                         [-254.1848809, 9.7133975]]], 'EPSG:4326', False),
            'APPLY_BORDER_NOISE_CORRECTION': False,
            'APPLY_SPECKLE_FILTERING': True,
            'SPECKLE_FILTER_FRAMEWORK':'MULTI',
            'SPECKLE_FILTER': 'LEE SIGMA',
            'SPECKLE_FILTER_KERNEL_SIZE': 7,
            'SPECKLE_FILTER_NR_OF_IMAGES':10,
            'APPLY_TERRAIN_FLATTENING': True,
            'DEM': ee.Image('USGS/SRTMGL1_003'),
            'TERRAIN_FLATTENING_MODEL': 'VOLUME',
            'TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER':0,
            'FORMAT': 'DB',
            'CLIP_TO_ROI': True,
            'SAVE_ASSET': True,
            'ASSET_ID': "users/sonleh96"
            }

APPLY_BORDER_NOISE_CORRECTION = params['APPLY_BORDER_NOISE_CORRECTION']
APPLY_TERRAIN_FLATTENING = params['APPLY_TERRAIN_FLATTENING']
APPLY_SPECKLE_FILTERING = params['APPLY_SPECKLE_FILTERING']
POLARIZATION = params['POLARIZATION']
PLATFORM_NUMBER = params['PLATFORM_NUMBER']
ORBIT = params['ORBIT']
ORBIT_NUM = params['ORBIT_NUM']
SPECKLE_FILTER_FRAMEWORK = params['SPECKLE_FILTER_FRAMEWORK']
SPECKLE_FILTER = params['SPECKLE_FILTER']
SPECKLE_FILTER_KERNEL_SIZE = params['SPECKLE_FILTER_KERNEL_SIZE']
SPECKLE_FILTER_NR_OF_IMAGES = params['SPECKLE_FILTER_NR_OF_IMAGES']
TERRAIN_FLATTENING_MODEL = params['TERRAIN_FLATTENING_MODEL']
DEM = params['DEM']
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = params['TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER']
FORMAT = params['FORMAT']
START_DATE = params['START_DATE']
STOP_DATE = params['STOP_DATE']
ROI = params['ROI']
CLIP_TO_ROI = params['CLIP_TO_ROI']
SAVE_ASSET = params['SAVE_ASSET']
ASSET_ID = params['ASSET_ID']

In [84]:
s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
        .filter(ee.Filter.eq('instrumentMode', 'IW'))\
        .filter(ee.Filter.eq('resolution_meters', 10)) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
        .filterDate(START_DATE, STOP_DATE) \
        .filterBounds(ROI)

s1 = s1.filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))
s1 = s1.filter(ee.Filter.eq('orbitProperties_pass', ORBIT))
s1 = s1.select(['VV', 'VH', 'angle'])
print('Number of images in collection: ', s1.size().getInfo())

Number of images in collection:  2


## Border Noise Correction

In [99]:
image = s1.first().clip(ROI)


image_viz_params = {
    'bands': ['VV'],
    'min': 0,
    'max': 1.0,
    'gamma': [1.1],
}

# Define a map centered on San Francisco Bay.
map_s1 = geemap.Map(center=[9.7257188, -254.1704284], zoom=15)

# Add the image layer to the map and display it.
map_s1.add_layer(image, image_viz_params, 'Raw S-1')
display(map_s1)

Map(center=[9.7257188, -254.1704284], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=S…

In [123]:
s1.first().select('VV').reduceRegion(ee.Reducer.min(), ROI).get('VV').getInfo()

0.002681118668988347

In [124]:
s1.first().select('VV').reduceRegion(ee.Reducer.max(), ROI).get('VV').getInfo()

5.494872093200684

In [170]:
# border noise correction

s1_1 = s1.map(bnc.f_mask_edges)
print('Additional border noise correction is completed')

Additional border noise correction is completed


In [125]:


image = s1_1.mean().clip(ROI)
image_viz_params = {
    'bands': ['VV'],
    'min': 0,
    'max': 1.0,
    'gamma': [1.1],\
}

# Define a map centered on San Francisco Bay.
map_s1 = geemap.Map(center=[9.7257188, -254.1704284], zoom=15)

# Add the image layer to the map and display it.
map_s1.add_layer(image, image_viz_params, 'Border noise corrected S-1')
display(map_s1)

SyntaxError: unterminated string literal (detected at line 8) (3946825603.py, line 8)

## Speckle Filtering

In [87]:
# Mono filter

s1_1_mono = ee.ImageCollection(sf.MonoTemporal_Filter(s1_1, SPECKLE_FILTER_KERNEL_SIZE, SPECKLE_FILTER))
print('Mono-temporal speckle filtering is completed')

image = s1_1_mono.first().clip(ROI)

image_viz_params = {
    'bands': ['VV'],
    'min': 0,
    'max': 1.0,
    'gamma': [1.1],
}

# mean = s1_1_mono.mean().select('VV').reduceRegion(
#     ee.Reducer.mean(), ROI).get('VV').getInfo()
# variance = s1_1_mono.mean().select('VV').reduceRegion(
#     ee.Reducer.variance(), ROI).get('VV').getInfo()
# print(mean**2 / variance)

# Define a map centered on San Francisco Bay.
map_s1 = geemap.Map(center=[9.7257188, -254.1704284], zoom=15)

# Add the image layer to the map and display it.
map_s1.add_layer(image, image_viz_params, 'mono')

# Multi filter

s1_1_multi = ee.ImageCollection(sf.MultiTemporal_Filter(s1_1, SPECKLE_FILTER_KERNEL_SIZE, SPECKLE_FILTER, SPECKLE_FILTER_NR_OF_IMAGES))
print('Multi-temporal speckle filtering is completed')

image = s1_1_multi.first().clip(ROI)
map_s1.add_layer(image, image_viz_params, 'multi')

# mean = s1_1_multi.first().select('VV').reduceRegion(
#     ee.Reducer.mean(), ROI).get('VV').getInfo()
# variance = s1_1_multi.first().select('VV').reduceRegion(
#     ee.Reducer.variance(), ROI).get('VV').getInfo()
# print(mean**2 / variance)

display(map_s1)

Mono-temporal speckle filtering is completed
Multi-temporal speckle filtering is completed


Map(center=[9.7257188, -254.1704284], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=S…

In [90]:
def calculate_ENL(image, band):
  mean = image.select(band).reduceRegion(
    ee.Reducer.mean(), ROI, 10, 'EPSG:4326').get(band).getInfo()
  variance = image.select(band).reduceRegion(
      ee.Reducer.variance(), ROI, 10, 'EPSG:4326').get(band).getInfo()
  return mean ** 2 / variance



In [91]:
print('ENL raw S-1 VV: ', calculate_ENL(s1.first(), 'VV'))
print('ENL bnc S-1 VV: ', calculate_ENL(s1_1.first(), 'VV'))
print('ENL mon S-1 VV: ', calculate_ENL(s1_1_mono.first(), 'VV'))
print('ENL mul S-1 VV: ', calculate_ENL(s1_1_multi.first(), 'VV'))
print('\n')
print('ENL raw S-1 VH: ', calculate_ENL(s1.first(), 'VH'))
print('ENL bnc S-1 VH: ', calculate_ENL(s1_1.first(), 'VH'))
print('ENL mon S-1 VH: ', calculate_ENL(s1_1_mono.first(), 'VH'))
print('ENL mul S-1 VH: ', calculate_ENL(s1_1_multi.first(), 'VH'))




ENL raw S-1 VV:  3.044818382476393
ENL bnc S-1 VV:  3.0448183824763806
ENL mon S-1 VV:  7.614111888856611
ENL mul S-1 VV:  4.773831481856531


ENL raw S-1 VH:  3.824243769731932
ENL bnc S-1 VH:  3.8242437697319187
ENL mon S-1 VH:  11.530043073150928
ENL mul S-1 VH:  7.9154061773326205


In [171]:
# Go with mono Lee Sigma

s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1_1, SPECKLE_FILTER_KERNEL_SIZE, SPECKLE_FILTER))

## Terrain correction

In [172]:
s1_1 = (trf.slope_correction(s1_1,
                             TERRAIN_FLATTENING_MODEL,
                             DEM,
                             TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))
print('Radiometric terrain normalization is completed')

Radiometric terrain normalization is completed


In [104]:
image = s1_1.first().clip(ROI)

image_viz_params = {
    'bands': ['VV'],
    'min': 0,
    'max': 1.0,
    'gamma': [1.1],
}

# Define a map centered on San Francisco Bay.
map_s1 = geemap.Map(center=[9.7257188, -254.1704284], zoom=15)

# Add the image layer to the map and display it.
map_s1.add_layer(image, image_viz_params, 'Terrain corrected')

display(map_s1)

Map(center=[9.7257188, -254.1704284], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=S…

In [106]:
print('ENL bnc mono tc S-1 VV: ', calculate_ENL(s1_1.first(), 'VV'))

print('ENL bnc mono tc S-1 VH: ', calculate_ENL(s1_1.first(), 'VH'))

ENL bnc mono tc S-1 VV:  6.494405440478238
ENL bnc mono tc S-1 VH:  9.690907480769058


In [118]:
s1_1.first().select('VV').reduceRegion(ee.Reducer.min(), ROI)

In [119]:
s1_1.first().select('VV').reduceRegion(ee.Reducer.max(), ROI)

In [1]:
s1_1_db = s1_1.map(helper.lin_to_db)

SyntaxError: invalid decimal literal (2886456483.py, line 1)

In [131]:
s1_1.first().select('VV').reduceRegion(ee.Reducer.min(), ROI).get('VV').getInfo()

-18.5871080790251

In [132]:
s1_1.first().select('VV').reduceRegion(ee.Reducer.max(), ROI).get('VV').getInfo()

8.126307432258617

## RVI

In [34]:
# def calculate_rvi_1(image):
#   numerator = image.select('VH').multiply(4)
#   denominator = image.select('VV').add(image.select('VH'))
#   rvi = numerator.divide(denominator).rename('RVI_1')

#   return image.addBands(rvi, None, True)


# def calculate_rvi_2(image):
#   q = image.select('VH').divide(image.select('VV'))
#   q = q.where(q.lt(0), 0)
#   q = q.where(q.gt(1), 1)
#   N = q.multiply(q.add(3))
#   D = (q.add(1)).multiply(q.add(1))
#   rvi = N.divide(D).rename('RVI_2')

#   return image.addBands(rvi, None, True)



def calculate_rvi(image):
  q = image.select('VH').divide(image.select('VV'))
  q = q.where(q.lt(0), 0)
  q = q.where(q.gt(1), 1)
  N = q.multiply(q.add(3))
  D = (q.add(1)).multiply(q.add(1))
  rvi = N.divide(D).rename('RVI')

  return image.addBands(rvi, None, True)



In [35]:
s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
        .filter(ee.Filter.eq('instrumentMode', 'IW'))\
        .filter(ee.Filter.eq('resolution_meters', 10)) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
        .filterDate('2020-01-01', '2020-02-01') \
        .filterBounds(ROI)

s1 = s1.filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))
s1 = s1.filter(ee.Filter.eq('orbitProperties_pass', ORBIT))
s1 = s1.select(['VV', 'VH', 'angle'])
print('Number of images in collection: ', s1.size().getInfo())

s1_1 = s1.map(bnc.f_mask_edges)
print('Additional border noise correction is complete')

s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1_1, SPECKLE_FILTER_KERNEL_SIZE, SPECKLE_FILTER))
print('Speckle filtering is complete')

s1_1 = (trf.slope_correction(s1_1,
                             TERRAIN_FLATTENING_MODEL,
                             DEM,
                             TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))
print('Radiometric terrain normalization is complete')

s1_1 = s1_1.map(calculate_rvi)
print('RVI computation is complete')

img_s1 = s1_1.mean()


Number of images in collection:  1
Additional border noise correction is complete
Speckle filtering is complete
Radiometric terrain normalization is complete
RVI computation is complete


In [8]:
# s1_1 = s1_1.map(calculate_rvi_1)

# print(s1_1.first().select('RVI_1').reduceRegion(ee.Reducer.min(), ROI).get('RVI_1').getInfo())
# print(s1_1.first().select('RVI_1').reduceRegion(ee.Reducer.max(), ROI).get('RVI_1').getInfo())

# s1_1 = s1_1.map(calculate_rvi)
# print(s1_1.first().select('RVI').reduceRegion(ee.Reducer.min(), ROI).get('RVI').getInfo())
# print(s1_1.first().select('RVI').reduceRegion(ee.Reducer.max(), ROI).get('RVI').getInfo())

0.021263371798107288
0.999981451151433


In [181]:
# s1_1_db = s1_1_db.map(calculate_rvi)
# print(s1_1_db.first().select('RVI').reduceRegion(ee.Reducer.min(), ROI).get('RVI').getInfo())
# print(s1_1_db.first().select('RVI').reduceRegion(ee.Reducer.max(), ROI).get('RVI').getInfo())

# converting to db does not work

-311.1486341044415
1.1249999998719495


In [ ]:
# image = s1_1.first().clip(ROI)

# image_viz_params = {
#     'bands': ['RVI'],
#     'min': 0,
#     'max': 1.0,
#     'gamma': [1.1],
# }
# map_s1 = geemap.Map(center=[9.7257188, -254.1704284], zoom=15)

# map_s1.add_layer(image, image_viz_params, 'RVI')

# display(map_s1)

## Applying Masks

In [10]:
# JAXA LULC VN dataset
# https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
jaxa = ee.Image('projects/bigdataobservatory-363204/assets/LULC_VN_small')
mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class
mask_jaxa = mask_jaxa.clip(ROI)


# Dynamic World LULC dataset
# https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1

dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')\
          .filterDate('2020-01-01', '2021-01-01')\
          .filterBounds(ROI)
mask_dw = dw.mode().clip(ROI).select('label').eq(4) # label == 4 is the "crops" class





In [ ]:
masked_dw_s1 = img_s1.updateMask(mask_dw)
masked_jaxa_s1 = img_s1.updateMask(mask_jaxa)
# both masks
masked_jaxa_dw_s1 = img_s1.updateMask(mask_dw).updateMask(mask_jaxa)

Map(center=[9.7257188, -254.1704284], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=S…

## Exporting

In [41]:
def extract_properties(feature):
  properties = {
      'Timestamp': '2020-01-01',
      'lon': feature.geometry().coordinates().get(0),
      'lat': feature.geometry().coordinates().get(1),
      'RVI': feature.get('RVI'),
      'CropType': 'Rice',
      'Country': 'Vietnam'
  }
  return ee.Feature(None, properties)

In [42]:
# feat_coll = masked_jaxa_dw_s1.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')
feat_coll = masked_jaxa_s1.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

processed_fc = feat_coll.map(extract_properties)

In [43]:
task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='RVI_sample_new',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'RVI', 'CropType', 'Country'],
                                     folder='outputs')

task.start()

In [46]:
df = pd.read_csv('/content/gdrive/MyDrive/Work/ADB/SAR/outputs/RVI_sample_new.csv')
df

,Timestamp,lon,lat,RVI,CropType,Country
0,2020-01-01,105.840809,9.713079,0.602129,Rice,Vietnam
1,2020-01-01,105.840899,9.713079,0.545812,Rice,Vietnam
2,2020-01-01,105.840989,9.713079,0.540959,Rice,Vietnam
3,2020-01-01,105.841079,9.713079,0.521530,Rice,Vietnam
4,2020-01-01,105.841169,9.713079,0.492648,Rice,Vietnam
...,...,...,...,...,...,...
4834,2020-01-01,105.842247,9.739579,0.563857,Rice,Vietnam
4835,2020-01-01,105.842336,9.739579,0.516607,Rice,Vietnam
4836,2020-01-01,105.842786,9.739579,0.482633,Rice,Vietnam
4837,2020-01-01,105.842875,9.739579,0.447109,Rice,Vietnam


# Aerosol

## MCD19A2.061 AOD

In [117]:
def get_qa_bits(image, start, end, new_name):
    # Compute the bits we need to extract
    pattern = 0
    for i in range(start, end + 1):
        pattern += 2 ** i

    # Return a single band image of the extracted QA bits, giving the band
    # a new name
    return image.select([0], [new_name]) \
                .bitwiseAnd(pattern) \
                .rightShift(start)

def mask_quality_flag(image):
    QA = image.select('AOD_QA')
    internal_quality = get_qa_bits(QA, 8, 11, 'internal_quality_flag')

    return image.updateMask(internal_quality.eq(0))

def mask_crop_map(image):
    global mask_jaxa
    return image.updateMask(mask_jaxa)

In [119]:
AOD = ee.ImageCollection("MODIS/006/MCD19A2_GRANULES")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .filterBounds(ROI)
#


AOD_masked = AOD.map(mask_quality_flag)
AOD_mean = AOD_masked.mean().select('Optical_Depth_047').multiply(0.001).rename('Optical_Depth_047')
AOD_reproject = AOD_mean.clip(ROI).reproject(crs='EPSG:4326', scale=10)



In [120]:
def extract_properties(feature):
  properties = {
      'Timestamp': '2020-01-01',
      'lon': feature.geometry().coordinates().get(0),
      'lat': feature.geometry().coordinates().get(1),
      'AOD': feature.get('Optical_Depth_047')
  }
  return ee.Feature(None, properties)

feat_coll = AOD_reproject.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='AOD_sample_masked_2',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'AOD'],
                                     folder='outputs')

task.start()

In [83]:
processed_fc.first().getInfo()

{'type': 'Feature',
 'geometry': None,
 'id': '0',
 'properties': {'AOD': 0.2895,
  'Coordinates': [105.840809332124, 9.713078925306531],
  'Timestamp': '2020-01-01'}}

In [84]:
task = ee.batch.Export.table.toDrive(collection=processed_fc.merge(processed_fc),
                                     description='AOD_sample_merged_test',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'Coordinates', 'AOD'],
                                     folder='outputs')

task.start()

In [104]:
df1 = pd.read_csv(os.path.join(dir, 'outputs/AOD_sample.csv'))
df1

,Timestamp,Coordinates,AOD
0,2020-01-01,"[105.840809332124, 9.713078925306531]",0.289500
1,2020-01-01,"[105.84089916365241, 9.713078925306531]",0.289500
2,2020-01-01,"[105.84098899518082, 9.713078925306531]",0.289500
3,2020-01-01,"[105.84107882670924, 9.713078925306531]",0.289500
4,2020-01-01,"[105.84116865823765, 9.713078925306531]",0.289500
...,...,...,...
91200,2020-01-01,"[105.8182616184926, 9.739938552301705]",0.291727
91201,2020-01-01,"[105.81835145002101, 9.739938552301705]",0.291727
91202,2020-01-01,"[105.81844128154941, 9.739938552301705]",0.291727
91203,2020-01-01,"[105.81853111307782, 9.739938552301705]",0.291727


In [74]:
df2 = pd.read_csv(os.path.join(dir, 'outputs/AOD_sample_merged.csv'))
len(df2)

182410

## CAMS

In [55]:
pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))

def mask_crop_map(image):
  return image.updateMask(mask_jaxa)

pm25 = pm25.map(mask_crop_map)

pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface').mean().rename('pm25')
pm25_clip = pm25_mean.clip(ROI)

feat_coll = pm25_clip.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'pm25': feature.get('pm25')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='pm25_sample_3',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'pm25'],
                                     folder='outputs')

task.start()

In [49]:
numberOfPixels = ee.List(pm25_reduced.values().get(0)).size()
numberOfPixels

In [51]:
bandNames = pm25_clip.bandNames()
bandNames

In [53]:
def create_feature(i):
    def set_band_value(band_name, feature):
        band_name = ee.String(band_name)
        pixel_value = ee.List(pm25_reduced.get(band_name)).get(i)
        return ee.Feature(feature).set(band_name, pixel_value)
    return bandNames.iterate(set_band_value, ee.Feature(ee.Geometry.Point([0, 0])))

# Map the function over a sequence
features = ee.FeatureCollection(
    ee.List.sequence(0, numberOfPixels.subtract(1)).map(create_feature)
)

In [71]:
pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))

pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                .mean()\
                .rename('pm25_mean')

masked_pm25 = pm25_mean.updateMask(mask_jaxa)

task = ee.batch.Export.image.toDrive(image=pm25_mean,
                                     description=f'pm25_sample',
                                     scale=1000,
                                     region=region,
                                     folder='outputs',
                                     fileNamePrefix=f'pm25_sample',
                                     crs='EPSG:4326',
                                     fileFormat='GeoTIFF')



task.start()

In [68]:
output = rasterstats.zonal_stats(os.path.join(dir, 'VN_pixels_small.geojson'),
                                 os.path.join(dir, 'outputs/pm25_sample.tif'),
                                 stats=['mean'],
                                 band_num=1)

/usr/local/lib/python3.10/dist-packages/rasterstats/io.py:328: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


In [69]:
output

[{'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.2135704885452014e-08},
 {'mean': 1.21357048

In [80]:
df_pm25 = pd.read_csv(os.path.join(dir, 'outputs/pm25_sample.csv'))
df_pm25

,Timestamp,lon,lat,pm25
0,2020-01-01,105.840809,9.713079,NaN
1,2020-01-01,105.840899,9.713079,NaN
2,2020-01-01,105.840989,9.713079,NaN
3,2020-01-01,105.841079,9.713079,NaN
4,2020-01-01,105.841169,9.713079,NaN
...,...,...,...,...
91200,2020-01-01,105.818262,9.739939,NaN
91201,2020-01-01,105.818351,9.739939,NaN
91202,2020-01-01,105.818441,9.739939,NaN
91203,2020-01-01,105.818531,9.739939,NaN


# Climate Conditions

In [57]:
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))

era5 = era5.map(mask_crop_map)

# era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
#             .filter(ee.Filter.date(START_DATE, STOP_DATE))

gldas = ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .filterBounds(ROI)

## ERA5

In [61]:
# Average temperature
mean_temp = era5.select('temperature_2m').mean()\
               .subtract(273.15)\
               .rename('mean_temperature')\
               .clip(ROI)
# Max temperature
max_temp = era5.select('temperature_2m').max()\
               .subtract(273.15)\
               .rename('max_temperature')\
               .clip(ROI)

# wind speed (m/s) and direction as angle phi (radians)
# https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
wind_u = era5.select('u_component_of_wind_10m').mean()
wind_v = era5.select('v_component_of_wind_10m').mean()
mean_wind_speed = ((wind_u.multiply(wind_u)).add(wind_v.multiply(wind_v))).sqrt().rename('mean_wind_speed')
mean_wind_speed = mean_wind_speed.clip(ROI)
mean_wind_phi = wind_u.divide(mean_wind_speed.abs()).asin().rename('mean_wind_direction')
mean_wind_phi = mean_wind_phi.clip(ROI)

# precipitation
prep = era5.select('total_precipitation_sum').sum()\
           .rename('total_precipitation')\
           .clip(ROI)

# Relative Humidity
# https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
b = 17.625
c = 243.04
mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
               .subtract(273.15)\
               .rename('avg_dew_temperature')\
               .clip(ROI)

# num = ee.Number(c).multiply(ee.Number(b)).multiply(mean_dew_temp.subtract(mean_temp))
num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
mean_rh = num.divide(deno).exp().rename('mean_relative_humidity')
mean_rh = mean_rh.clip(ROI)

climate = mean_temp.addBands(max_temp)\
                   .addBands(prep)\
                   .addBands(mean_wind_speed)\
                   .addBands(mean_wind_phi)\
                   .addBands(mean_rh)\
                   .reproject(crs='EPSG:4326', scale=10)

feat_coll = climate.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'mean_temperature': feature.get('mean_temperature'),
        'max_temperature': feature.get('max_temperature'),
        'total_precipitation': feature.get('total_precipitation'),
        'mean_wind_speed': feature.get('mean_wind_speed'),
        'mean_wind_direction': feature.get('mean_wind_direction'),
        'mean_relative_humidity': feature.get('mean_relative_humidity')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='climate_sample_era5',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'mean_temperature',
                                                'max_temperature', 'total_precipitation', 'mean_wind_speed',
                                                'mean_wind_direction', 'mean_relative_humidity'],
                                     folder='outputs')


task.start()


In [58]:
# Average temperature
mean_temp = era5.select('temperature_2m').mean().rename('mean_temperature')

# Max temperature
max_temp = era5.select('temperature_2m').max().rename('max_temperature')

# Dew point temperature
mean_dew_temp = era5.select('dewpoint_temperature_2m').mean().rename('mean_dew_temperature')

# wind speed (m/s) and direction as angle phi (radians)
# https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
wind_u = era5.select('u_component_of_wind_10m').mean().rename('u_wind')
wind_v = era5.select('v_component_of_wind_10m').mean().rename('v_wind')

# precipitation
prep = era5.select('total_precipitation_sum').sum()\
           .rename('total_precipitation')\
           .clip(ROI).reproject(crs='EPSG:4326', scale=10)

climate = mean_temp.addBands(max_temp)\
                   .addBands(mean_dew_temp)\
                   .addBands(prep)\
                   .addBands(wind_u)\
                   .addBands(wind_v)
climate_reproject = climate.clip(ROI).reproject(crs='EPSG:4326', scale=10)


feat_coll = climate_reproject.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'mean_temperature': feature.get('mean_temperature'),
        'max_temperature': feature.get('max_temperature'),
        'mean_dew_temperature': feature.get('mean_dew_temperature'),
        'total_precipitation': feature.get('total_precipitation'),
        'u_wind': feature.get('u_wind'),
        'v_wind': feature.get('v_wind')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='climate_sample',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'mean_temperature',
                                                'max_temperature', 'mean_dew_temperature', 'total_precipitation',
                                                'u_wind', 'v_wind'],
                                     folder='outputs')

task.start()

In [58]:
# # Average temperature
# mean_temp = era5.select('temperature_2m').mean()

# # Max temperature
# max_temp = era5.select('temperature_2m').max()

# # Mean dew teamperature
# mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
#                .rename('avg_dew_temperature')\

# # wind speed (m/s) and direction as angle phi (radians)
# # https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
# wind_u = era5.select('u_component_of_wind_10m').mean().rename('mean_u_wind')
# wind_v = era5.select('v_component_of_wind_10m').mean().rename('mean_v_wind')


# # precipitation
# prep = era5.select('total_precipitation_sum').sum()\
#            .rename('total_precipitation')

# climate = mean_temp.addBands(max_temp)\
#                    .addBands(mean_dew_temp)\
#                    .addBands(prep)\
#                    .addBands(wind_u)\
#                    .addBands(wind_v)

# task = ee.batch.Export.image.toDrive(image=climate,
#                                      description=f'climate_sample',
#                                      scale=1000,
#                                      region=region,
#                                      folder='outputs',
#                                      fileNamePrefix=f'climate_sample',
#                                      crs='EPSG:4326',
#                                      fileFormat='GeoTIFF')



# task.start()

In [82]:
cli_df = pd.read_csv(os.path.join(dir, 'outputs/climate_sample.csv'))
cli_df

,Timestamp,lon,lat,mean_temperature,max_temperature,mean_dew_temperature,total_precipitation,u_wind,v_wind
0,2020-01-01,105.840809,9.713079,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01,105.840899,9.713079,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01,105.840989,9.713079,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01,105.841079,9.713079,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01,105.841169,9.713079,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
91200,2020-01-01,105.818262,9.739939,NaN,NaN,NaN,NaN,NaN,NaN
91201,2020-01-01,105.818351,9.739939,NaN,NaN,NaN,NaN,NaN,NaN
91202,2020-01-01,105.818441,9.739939,NaN,NaN,NaN,NaN,NaN,NaN
91203,2020-01-01,105.818531,9.739939,NaN,NaN,NaN,NaN,NaN,NaN


## GLDAS 2.1

In [94]:
# Average temperature
mean_temp = gldas.select('Tair_f_inst').mean()\
                 .subtract(273.15)\
                 .rename('mean_temperature')\
                 .clip(ROI).reproject(crs='EPSG:4326', scale=10)

# Max temperature
max_temp = gldas.select('Tair_f_inst').max()\
                 .subtract(273.15)\
                 .rename('max_temperature')\
                 .clip(ROI).reproject(crs='EPSG:4326', scale=10)

# wind speed (m/s)
wind = gldas.select('Wind_f_inst').mean()\
            .rename('mean_wind_speed')\
            .clip(ROI).reproject(crs='EPSG:4326', scale=10)


# https://earthscience.stackexchange.com/questions/24156/era5-single-level-calculate-relative-humidity
# specific humidity
e = gldas.select('Qair_f_inst').mean()

# saturation point
# a = ee.Image.constant(6.1078).clip(ROI).reproject(crs='EPSG:4326', scale=10)
# b = ee.Image.constant(17.1).clip(ROI).reproject(crs='EPSG:4326', scale=10)
# c = ee.Image.constant(235).clip(ROI).reproject(crs='EPSG:4326', scale=10)
num = mean_temp.multiply(17.1)
deno = mean_temp.add(235)
es = num.divide(deno).exp().multiply(6.1078)

# relative humidity
rh = e.divide(es)

climate = mean_temp.addBands(max_temp)\
                   .addBands(mean_wind_speed)\
                   .addBands(mean_rh)

# climate_masked = climate.updateMask(mask_jaxa)

feat_coll = climate.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'mean_temperature': feat_coll.get('mean_temperature'),
        'max_temperature': feat_coll.get('max_temperature'),
        'mean_wind_speed': feat_coll.get('mean_wind_speed'),
        'mean_relative_humidity': feat_coll.get('mean_relative_humidity')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='climate_sample_gldas',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'mean_temperature',
                                                'max_temperature', 'mean_wind_speed', 'mean_relative_humidity'],
                                     folder='outputs')


task.start()

In [103]:
# Average temperature
mean_temp = gldas.select('Tair_f_inst').mean()\
                 .subtract(273.15)\
                 .rename('mean_temperature')
mean_temp = mean_temp.reproject(crs=mean_temp.projection(), scale=10).clip(ROI)

# feat_coll = mean_temp.sample(factor=1, region=ROI, geometries=True, projection='EPSG:4326', scale=10)
feat_coll = mean_temp

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'mean_temperature': feat_coll.get('mean_temperature')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='climate_sample_gldas_3',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat', 'mean_temperature'],
                                     folder='outputs')
task.start()

# FIRMS

In [62]:
firms = ee.ImageCollection('FIRMS').filter(ee.Filter.date(START_DATE, STOP_DATE))

firms = firms.map(mask_crop_map)

def mask_fire_confidence_flag(image):
    confidence_mask = image.select('confidence')

    return image.updateMask(confidence_mask.gt(50))

firms = firms.map(mask_fire_confidence_flag)
mean_fire = firms.select('T21').mean()\
                 .rename('mean_brightness_temperature')\
                 .clip(ROI)
max_fire = firms.select('T21').max()\
                .rename('max_brightness_temperature')\
                .clip(ROI)

fires = mean_fire.addBands(max_fire).reproject(crs='EPSG:4326', scale=10)
feat_coll = fires.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'lon': feature.geometry().coordinates().get(0),
        'lat': feature.geometry().coordinates().get(1),
        'mean_brightness_temp': feature.get('mean_brightness_temperature'),
        'max_brightness_temp': feature.get('max_brightness_temperature')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='firms_sample',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'lon', 'lat',
                                                'mean_brightness_temperature',
                                                'max_brightness_temperature'],
                                     folder='outputs')


task.start()

In [63]:
dataset = ee.ImageCollection('FIRMS').filter(
    ee.Filter.date('2018-08-01', '2018-08-10'));
fires = dataset.select('T21');
type(fires)

ee.imagecollection.ImageCollection

# Luminosity

In [128]:
bm = ee.Image('projects/bigdataobservatory-363204/assets/VNM_bm_2020_01').clip(ROI).reproject(crs='EPSG:4326', scale=10)

feat_coll = bm.sample(factor=1, region=ROI, geometries=True, scale=10, projection='EPSG:4326')

def extract_properties(feature):
  properties = {
        'Timestamp': '2020-01-01',
        'Coordinates': feature.geometry().coordinates(),
        'luminosity': feature.get('luminosity')
    }
  return ee.Feature(None, properties)

processed_fc = feat_coll.map(extract_properties)

task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                     description='bm_sample',
                                     fileFormat='CSV',
                                     selectors=['Timestamp', 'Coordinates', 'luminosity'],
                                     folder='outputs')


# task.start()

In [129]:
feat_coll

/usr/local/lib/python3.10/dist-packages/eerepr/repr.py:57: UserWarning: Getting info failed with: 'Collection query aborted after accumulating over 5000 elements.'. Falling back to string repr.
  warn(f"Getting info failed with: '{e}'. Falling back to string repr.")


In [133]:
m = gdal.Warp(os.path.join(dir, 'LULC_VN_cropped.tif'),
              os.path.join(dir, 'LULC_VN.tif'),
              cutlineDSName='/content/gdrive/MyDrive/Work/WB/GPBP/GED/shapefiles/gadm41_VNM_0.json',
              cropToCutline=True)
m = None

# Combined

## Functions

In [ ]:
def calculate_rvi(image):
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI_mean')

  return rvi

def get_qa_bits(image, start, end):
    # Compute the bits we need to extract
    pattern = 0
    for i in range(start, end + 1):
        pattern += 2**i

    # Return a single band image of the extracted QA bits, giving the band
    # a new name
    return image.select(0) \
                .bitwiseAnd(pattern) \
                .rightShift(start)

def mask_quality(image):
    QA = image.select('AOD_QA')
    internal_quality = get_qa_bits(QA, 8, 11)

    return image.updateMask(internal_quality.eq(0))

def mask_crop_map(image):
    return image.updateMask(mask_jaxa)

def mask_fire_confidence_flag(image):
    confidence_mask = image.select('confidence')

    return image.updateMask(confidence_mask.gt(50))

def fill_nodata(image):
    image_nodata = image.focal_mean(1, 'square', 'pixels', 20)
    return image_nodata.blend(image)



## Params

In [94]:
# Sample
# ROI = ee.Geometry.Polygon([[[-254.1842336, 9.739978],
#                             [-254.1569875, 9.739608],
#                             [-254.1567476, 9.713047],
#                             [-254.1848809, 9.7133975]]], 'EPSG:4326', False)

# Vietnam
# ROI = ee.Geometry.Polygon([[[-250.4296884, 23.9818253],
#                             [-250.0341806, 7.4493043],
#                             [-258.2958993, 8.0152481],
#                             [-257.9003915, 23.901518]]], 'EPSG:4326', False)

# South VN
# ROI = ee.Geometry.Polygon([[[104.6777344,9.055886],
#                             [105.4742432,9.0396158],
#                             [105.4797363,8.529451],
#                             [104.6942139,8.5457436]]], 'EPSG:4326', False)

# ROI = ee.Geometry.Polygon([[[105.333, 9.667],
#                             [105.333 , 10],
#                             [105, 10],
#                             [105, 9.667]]], 'EPSG:4326', False)

# South VN BBox
# ROI = ee.Geometry.BBox(105.0, 9.5, 105.5, 10.0)\
#                  .bounds(proj='EPSG:4326')

# Vietnam Feature Collection
# ROI = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.inList('ADM0_NAME', ['Viet Nam']))

with open('JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

crop_type = 'Rice'
country = 'VN'
crs = 'EPSG:4326'
scale=100



START_YM = '2017-01'
END_YM = '2017-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'MULTI'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 3 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0
CLIP_TO_ROI = True


FileNotFoundError: [Errno 2] No such file or directory: 'JAXA_subtiles.json'

In [19]:
tiles = list(jaxa_subtiles.keys())
tiles[33:]

['N18E105_0_0',
 'N18E105_0_1',
 'N18E105_1_1',
 'N18E106_1_0',
 'N19E105_0_0',
 'N19E105_0_1',
 'N19E105_1_0',
 'N19E105_1_1',
 'N20E105_0_1',
 'N20E105_1_1',
 'N20E106_0_0',
 'N20E106_0_1',
 'N20E106_1_0',
 'N20E106_1_1',
 'N21E105_0_0',
 'N21E105_0_1',
 'N21E105_1_0',
 'N21E105_1_1',
 'N21E106_0_0',
 'N21E106_0_1',
 'N21E106_1_0',
 'N21E106_1_1',
 'N21E107_0_1',
 'N21E107_1_0',
 'N21E107_1_1']

## Full Extraction

In [24]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# for tile in list(jaxa_subtiles.keys())[1:16]:
for tile in list(jaxa_subtiles.keys())[33:]:

  print(f"  {tile}")
  bbox = jaxa_subtiles[tile]
  ROI = ee.Geometry.BBox(bbox[0],
                         bbox[1],
                         bbox[2],
                         bbox[3])\
                   .bounds(proj=crs)
  
  # JAXA LULC VN dataset
  # https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
  jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
  mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class
                   

  for month in months[:]:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')



    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
            .filter(ee.Filter.eq('instrumentMode', 'IW'))\
            .filter(ee.Filter.eq('resolution_meters', 10)) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
            .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
            .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
            .filterDate(START_DATE, STOP_DATE)\
            .select(['VV', 'VH', 'angle'])

      # Border noise correction
    # s1_1 = s1.map(bnc.f_mask_edges)

      # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                    SPECKLE_FILTER_KERNEL_SIZE,
                                                    SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                TERRAIN_FLATTENING_MODEL,
                                DEM,
                                TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

      # RVI
    s1_1 = s1_1.map(calculate_rvi)


    rvi = s1_1.mean()\
              .updateMask(mask_jaxa)\
              .reproject(crs=crs, scale=scale)\
              .clip(ROI)\



    # CAMS PM2.5
    pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

    pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .mean()\
                    .multiply(1E9)\
                    .rename('PM25_mean')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
    
    pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)  
    
    # Sentinel-5P
    # no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")\
    #         .filter(ee.Filter.date(START_DATE, STOP_DATE))\
    #         .map(mask_crop_map)\
    #         .filterBounds(ROI)
    # no2_mean = no2.select('NO2_column_number_density')\
    #               .mean()\
    #               .rename('NO2_mean')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    # no2_max = no2.select('NO2_column_number_density')\
    #               .max()\
    #               .rename('NO2_max')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    
    # o3 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_O3")\
    #         .filter(ee.Filter.date(START_DATE, STOP_DATE))\
    #         .map(mask_crop_map)\
    #         .filterBounds(ROI)
    # o3_mean = o3.select('O3_column_number_density')\
    #               .mean()\
    #               .rename('O3_mean')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    # o3_max = o3.select('O3_column_number_density')\
    #               .max()\
    #               .rename('O3_max')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    
    # so2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_SO2")\
    #         .filter(ee.Filter.date(START_DATE, STOP_DATE))\
    #         .map(mask_crop_map)\
    #         .filterBounds(ROI)
    # so2_mean = so2.select('SO2_column_number_density')\
    #               .mean()\
    #               .rename('SO2_mean')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    # so2_max = so2.select('SO2_column_number_density')\
    #               .max()\
    #               .rename('SO2_max')\
    #               .clip(ROI)\
    #               .reproject(crs=crs, scale=scale)
    

    # ERA5
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

      # Average temperature
    mean_temp = era5.select('temperature_2m').mean()\
                  .subtract(273.15)\
                  .rename('Temperature_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)

      # Max temperature
    max_temp = era5.select('temperature_2m').max()\
                  .subtract(273.15)\
                  .rename('Temperature_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)

      # wind speed (m/s) and direction as angle phi (radians)
      # https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
    # mean_wind_u = era5.select('u_component_of_wind_10m').mean()
    # mean_wind_v = era5.select('v_component_of_wind_10m').mean()
    # mean_wind_speed = ((mean_wind_u.multiply(mean_wind_u)).add(mean_wind_v.multiply(mean_wind_v))).sqrt()\
    #                               .rename('Wind_speed_mean')\
    #                               .clip(ROI)\
    #                               .reproject(crs=crs, scale=scale)
    # mean_wind_phi = mean_wind_u.divide(mean_wind_speed.abs()).asin()\
    #                           .rename('Wind_direction_mean')\
    #                           .clip(ROI)\
    #                           .reproject(crs=crs, scale=scale)
    
    # max_wind_u = era5.select('u_component_of_wind_10m').max()
    # max_wind_v = era5.select('v_component_of_wind_10m').max()
    # max_wind_speed = ((max_wind_u.multiply(max_wind_u)).add(max_wind_v.multiply(max_wind_v))).sqrt()\
    #                               .rename('Wind_speed_max')\
    #                               .clip(ROI)\
    #                               .reproject(crs=crs, scale=scale)
    # max_wind_phi = max_wind_u.divide(max_wind_speed.abs()).asin()\
    #                          .rename('Wind_direction_max')\
    #                          .clip(ROI)\
    #                          .reproject(crs='EPSG:4326', scale=10)


    #   precipitation
    prep = era5.select('total_precipitation_sum').sum()\
              .rename('Precipitation_sum')\
              .clip(ROI)\
              .reproject(crs=crs, scale=scale)

      # Relative Humidity
      # https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
    b = 17.625
    c = 243.04
    mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
                        .subtract(273.15)\
                        .rename('avg_dew_temperature')\
                        .clip(ROI)\
                        .reproject(crs=crs, scale=scale)

    num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
    deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
    mean_rh = num.divide(deno).exp()\
                .rename('Rel_humidity_mean')

    
    # Black Marble
    bm = ee.Image(f"projects/ee-sonle96/assets/BM_VN_processed/VNM_bm_{month.replace('-', '_')}")\
          .select('b1')\
          .rename('Luminosity')\
          .clip(ROI)\
          .updateMask(mask_jaxa)\
          .reproject(crs=crs, scale=scale)
    
    # Elevation
    # negative values indicate elevation is below mean sea level  
    dem = ee.Image('USGS/SRTMGL1_003')\
            .select('elevation')\
            .updateMask(mask_jaxa)\
            .clip(ROI)\
            .reproject(crs=crs, scale=scale)
    

    # Combined

    # combined = rvi.addBands(pm25_mean)\
    #               .addBands(pm25_max)\
    #               .addBands(no2_mean)\
    #               .addBands(no2_max)\
    #               .addBands(o3_mean)\
    #               .addBands(o3_max)\
    #               .addBands(so2_mean)\
    #               .addBands(so2_max)\
    #               .addBands(mean_temp)\
    #               .addBands(max_temp)\
    #               .addBands(mean_wind_speed)\
    #               .addBands(mean_wind_phi)\
    #               .addBands(max_wind_speed)\
    #               .addBands(prep)\
    #               .addBands(mean_rh)\
    #               .addBands(bm)\
    #               .addBands(dem)
    
    combined = rvi.addBands(pm25_mean)\
                  .addBands(pm25_max)\
                  .addBands(mean_temp)\
                  .addBands(max_temp)\
                  .addBands(prep)\
                  .addBands(mean_rh)\
                  .addBands(bm)\
                  .addBands(dem)


    feat_coll = combined.sample(factor=1,
                                region=ROI,
                                geometries=True,
                                scale=scale,
                                projection=crs)



    def extract_properties(feature):
      properties = {
          'Timestamp': START_DATE,
          'lon': feature.geometry().coordinates().get(0),
          'lat': feature.geometry().coordinates().get(1),
          'CropType': crop_type,
          'Country': country,
          'RVI_mean': feature.get('RVI_mean'),
          'PM25_mean': feature.get('PM25_mean'),
          'PM25_max': feature.get('PM25_max'),
          # 'NO2_mean': feature.get('NO2_mean'),
          # 'NO2_max': feature.get('NO2_max'),
          # 'O3_mean': feature.get('O3_mean'),
          # 'O3_max': feature.get('O3_max'),
          # 'SO2_mean': feature.get('SO2_mean'),
          # 'SO2_max': feature.get('SO2_max'),
          'Temperature_mean': feature.get('Temperature_mean'),
          'Temperature_max': feature.get('Temperature_max'),
          # 'Wind_speed_mean': feature.get('Wind_speed_mean'),
          # 'Wind_direction_mean': feature.get('Wind_direction_mean'),
          # 'Wind_speed_max': feature.get('Wind_speed_max'),
          'Rel_humidity_mean': feature.get('Rel_humidity_mean'),
          'Precipitation_sum': feature.get('Precipitation_sum'),
          'Luminosity': feature.get('Luminosity'),
          'elevation': feature.get('elevation')
      }
      return ee.Feature(None, properties)

    processed_fc = feat_coll.map(extract_properties)

    # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
    #       'RVI_mean', 'PM25_mean', 'PM25_max', 'NO2_mean', 'NO2_max',
    #       'O3_mean', 'O3_max', 'SO2_mean', 'SO2_max', 'Temperature_mean',
    #       'Temperature_max', 'Wind_speed_mean', 'Wind_direction_mean',
    #       'Wind_speed_max', 
    #       'Rel_humidity_mean', 'Precipitation_sum', 'Luminosity', 'elevation']
    
    bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
             'RVI_mean', 'PM25_mean', 'PM25_max', 'Temperature_mean',
             'Temperature_max', 'Rel_humidity_mean', 'Precipitation_sum', 
             'Luminosity', 'elevation']
    
    task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                        description=f'{tile}_{scale}m_{START_DATE}',
                                        fileFormat='CSV',
                                        selectors=bands,
                                        folder='final_north')

    task.start()


CRS: EPSG:4326
scale: 100
Start: 2017-01. End: 2017-12
  N18E105_0_0
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
  N18E105_0_1
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
  N18E105_1_1
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
  N18E106_1_0
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
  N19E105_0_0
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
  N19E105_0_1
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-

## Category Pollutants

In [20]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# for tile in list(jaxa_subtiles.keys())[1:16]:
for tile in list(jaxa_subtiles.keys())[33:]:

  print(f"  {tile}")
  bbox = jaxa_subtiles[tile]
  ROI = ee.Geometry.BBox(bbox[0],
                         bbox[1],
                         bbox[2],
                         bbox[3])\
                   .bounds(proj=crs)
  
  # JAXA LULC VN dataset
  # https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
  jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
  mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class
                   

  for month in months[:]:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')


    
    # Sentinel-5P
    no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .map(mask_crop_map)\
            .filterBounds(ROI)
    no2_mean = no2.select('tropospheric_NO2_column_number_density')\
                  .mean()\
                  .rename('NO2_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    no2_max = no2.select('tropospheric_NO2_column_number_density')\
                  .max()\
                  .rename('NO2_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    
    o3 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_O3_TCL")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .map(mask_crop_map)\
            .filterBounds(ROI)
    o3_mean = o3.select('ozone_tropospheric_vertical_column')\
                  .mean()\
                  .rename('O3_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    o3_max = o3.select('ozone_tropospheric_vertical_column')\
                  .max()\
                  .rename('O3_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    
    


    combined = no2_mean.addBands(no2_max)\
                       .addBands(o3_mean)\
                       .addBands(o3_max)\


    feat_coll = combined.sample(factor=1,
                                region=ROI,
                                geometries=True,
                                scale=scale,
                                projection=crs)



    def extract_properties(feature):
      properties = {
          'Timestamp': START_DATE,
          'lon': feature.geometry().coordinates().get(0),
          'lat': feature.geometry().coordinates().get(1),
          'NO2_mean': feature.get('NO2_mean'),
          'NO2_max': feature.get('NO2_max'),
          'O3_mean': feature.get('O3_mean'),
          'O3_max': feature.get('O3_max')
      }
      return ee.Feature(None, properties)

    processed_fc = feat_coll.map(extract_properties)

    bands = ['Timestamp', 'lon', 'lat', 'NO2_mean', 'NO2_max',
             'O3_mean', 'O3_max']
    
    task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                        description=f'{tile}_{scale}m_{START_DATE}',
                                        fileFormat='CSV',
                                        selectors=bands,
                                        folder='tropo_north')

    task.start()


CRS: EPSG:4326
scale: 100
Start: 2018-01. End: 2022-12
  N18E105_0_0
    2018-01
    2018-02
    2018-03
    2018-04
    2018-05
    2018-06
    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2019-05
    2019-06
    2019-07
    2019-08
    2019-09
    2019-10
    2019-11
    2019-12
    2020-01
    2020-02
    2020-03
    2020-04
    2020-05
    2020-06
    2020-07
    2020-08
    2020-09
    2020-10
    2020-11
    2020-12
    2021-01
    2021-02
    2021-03
    2021-04
    2021-05
    2021-06
    2021-07
    2021-08
    2021-09
    2021-10
    2021-11
    2021-12
    2022-01
    2022-02
    2022-03
    2022-04
    2022-05
    2022-06
    2022-07
    2022-08
    2022-09
    2022-10
    2022-11
    2022-12
  N18E105_0_1
    2018-01
    2018-02
    2018-03
    2018-04
    2018-05
    2018-06
    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2

In [6]:
crs = 'EPSG:4326'
scale=1000

START_YM = '2018-07'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

region = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.inList('ADM0_NAME', ['Indonesia'])).geometry()

for month in months:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')
                                            
    # Sentinel-5P
    no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .select('tropospheric_NO2_column_number_density')

    no2_mean = no2.mean().rename('NO2_mean').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    no2_max = no2.max().rename('NO2_max').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    
    o3 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_O3_TCL")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .select('ozone_tropospheric_vertical_column')


    o3_mean = o3.mean().rename('O3_mean').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    o3_max = o3.max().rename('O3_max').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )


    properties = {
            'Timestamp': START_DATE,
            'NO2_mean': no2_mean.get('NO2_mean'),
            'NO2_max': no2_max.get('NO2_max'),
            'O3_mean': o3_mean.get('O3_mean'),
            'O3_max': o3_max.get('O3_max')
        }
    
    export = ee.FeatureCollection(ee.Feature(None, properties))

    task = ee.batch.Export.table.toDrive(collection=export,
                                            description=f'VNM_FIRMS_{START_DATE}',
                                            fileFormat='CSV',
                                            selectors=['Timestamp', 'NO2_mean', 'NO2_max', 'O3_mean', 'O3_max'],
                                            folder='NO2_O3_IND')

    task.start()

    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2019-05
    2019-06
    2019-07
    2019-08
    2019-09
    2019-10
    2019-11
    2019-12
    2020-01
    2020-02
    2020-03
    2020-04
    2020-05
    2020-06
    2020-07
    2020-08
    2020-09
    2020-10
    2020-11
    2020-12
    2021-01
    2021-02
    2021-03
    2021-04
    2021-05
    2021-06
    2021-07
    2021-08
    2021-09
    2021-10
    2021-11
    2021-12
    2022-01
    2022-02
    2022-03
    2022-04
    2022-05
    2022-06
    2022-07
    2022-08
    2022-09
    2022-10
    2022-11
    2022-12


## EE Dataset Upload

In [9]:
!earthengine authenticate

Authenticate: Credentials already exist.  Use --force to refresh.


In [10]:
paths = glob.glob('/content/gdrive/MyDrive/Work/WB/GPBP/data/VNM/luminosity/*.tif')
paths[0][paths[0].rindex('/')+1:paths[0].rindex('.')]

IndexError: list index out of range

In [ ]:
for path in paths[:]:
  ee_id = path[path.rindex('/')+1:path.rindex('.')]
  gcb_id = path[path.rindex('/')+1:]
  
  print(ee_id, gcb_id)
  !earthengine upload image --asset_id=projects/ee-sonle96/assets/BM_VN_processed/{ee_id} gs://adb-sar/black-marble-vn/{gcb_id}

In [ ]:
!earthengine upload image --asset_id=projects/ee-sonle96/assets/LULC_VN gs://adb-sar/lulc/LULC_VN_clipped.tif

## FIRMS

In [5]:
crs = 'EPSG:4326'
scale=1000

START_YM = '2016-12'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

region = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.inList('ADM0_NAME', ['Viet Nam'])).geometry()

for month in months:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')
                                            
    # FIRMS
    firms = ee.ImageCollection('FIRMS')\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .select('T21')

    firms_mean = firms.mean().rename('Mean_T21')
    firms_max = firms.max().rename('Max_T21')

    fire_mean = firms_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    fire_max = firms_max.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    # fire = ee.FeatureCollection([fire_mean, fire_max])

    # def shape_export(feature):
    #     properties = {
    #         'Timestamp': START_DATE,
    #         'Mean_T21': feature.get('Mean_T21'),
    #         'Max_T21': feature.get('Max_T21')
    #     }
    #     return ee.Feature(None, properties)

    # export = fire.map(shape_export)
    properties = {
            'Timestamp': START_DATE,
            'Mean_T21': fire_mean.get('Mean_T21'),
            'Max_T21': fire_max.get('Max_T21')
        }
    export = ee.FeatureCollection(ee.Feature(None, properties))

    task = ee.batch.Export.table.toDrive(collection=export,
                                            description=f'VNM_FIRMS_{START_DATE}',
                                            fileFormat='CSV',
                                            selectors=['Timestamp', 'Mean_T21', 'Max_T21'],
                                            folder='FIRMS_VNM')

    task.start()

    2016-12
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
    2018-01
    2018-02
    2018-03
    2018-04
    2018-05
    2018-06
    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2019-05
    2019-06
    2019-07
    2019-08
    2019-09
    2019-10
    2019-11
    2019-12
    2020-01
    2020-02
    2020-03
    2020-04
    2020-05
    2020-06
    2020-07
    2020-08
    2020-09
    2020-10
    2020-11
    2020-12
    2021-01
    2021-02
    2021-03
    2021-04
    2021-05
    2021-06
    2021-07
    2021-08
    2021-09
    2021-10
    2021-11
    2021-12
    2022-01
    2022-02
    2022-03
    2022-04
    2022-05
    2022-06
    2022-07
    2022-08
    2022-09
    2022-10
    2022-11
    2022-12


## NVDI, LAI

In [5]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("EVI")

    image = image.addBands(EVI)

    return(image)

def getLAI(image):
    LAI = image.expression(
        '(3.618*EVI - 0.118)', {
            'EVI': image.select('EVI')
        }).rename("LAI_mean")
    image = image.addBands(LAI)

    return(image)

def getNDVI(image):
    
    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI_mean")

    return(ndvi)

# def mask_wheat_map(img):
    
#     wheat_class_mask = img.select('classification').gt(50)
#     wheat_confi_mask = img.select('confidence').gt(50)
#     return img.updateMask(wheat_class_mask).updateMask(wheat_confi_mask)




In [6]:
with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

crop_type = 'Maize'
country = 'VN'
crs = 'EPSG:4326'
scale=100



START_YM = '2017-01'
# END_YM = '2017-02'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_8412\3648787058.py:1: SyntaxWarning: invalid escape sequence '\W'
  with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:


In [ ]:
# chosen_tiles = [k for k in jaxa_subtiles.keys() if 'N10E105' in k] + [k for k in jaxa_subtiles.keys() if 'N09E105' in k] + ['N10E106_0_0', 'N10E106_0_1']
chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
# chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
#                 'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']
chosen_tiles


['N10E105_0_1',
 'N10E105_1_1',
 'N10E106_0_1',
 'N10E105_1_0',
 'N09E105_1_0',
 'N10E104_1_1',
 'N09E105_0_0',
 'N09E105_1_0',
 'N09E105_0_1',
 'N09E105_1_1',
 'N10E105_0_0',
 'N10E106_0_0']

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

def mask_crop_map(image):
    return image.updateMask(wheat_mask)

for tile in chosen_tiles[4:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
        # Sentinel-2 1C -> radiometrically calibrated
        ndvi = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)\
            .map(getNDVI)
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = ndvi.mean()\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)\
                
        ndvi_min = ndvi.min()\
                .rename('NDVI_min')\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)
        
        ndvi_max = ndvi.max()\
                .rename('NDVI_max')\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)
        
        pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

        pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                        .mean()\
                        .multiply(1E9)\
                        .rename('PM25_mean')\
                        .clip(ROI)\
                        .reproject(crs='EPSG:4326', scale=100)
        
        pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
        
        # ERA5
        era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
                .filter(ee.Filter.date(START_DATE, STOP_DATE))\
                .map(mask_crop_map)

        # Average temperature
        mean_temp = era5.select('temperature_2m').mean()\
                    .subtract(273.15)\
                    .rename('Temperature_mean')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)

        # Max temperature
        max_temp = era5.select('temperature_2m').max()\
                    .subtract(273.15)\
                    .rename('Temperature_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)

        #   precipitation
        prep = era5.select('total_precipitation_sum').sum()\
                .rename('Precipitation_sum')\
                .clip(ROI)\
                .reproject(crs=crs, scale=scale)

        # Relative Humidity
        # https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
        b = 17.625
        c = 243.04
        mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
                            .subtract(273.15)\
                            .rename('avg_dew_temperature')\
                            .clip(ROI)\
                            .reproject(crs=crs, scale=scale)

        num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
        deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
        mean_rh = num.divide(deno).exp()\
                    .rename('Rel_humidity_mean')

        
        # Black Marble
        bm = ee.Image(f"projects/ee-sonle96/assets/BM_VN_processed/VNM_bm_{month.replace('-', '_')}")\
            .select('b1')\
            .rename('Luminosity')\
            .clip(ROI)\
            .reproject(crs=crs, scale=scale)\
            .updateMask(maize_mask)
            
        
        # Elevation
        # negative values indicate elevation is below mean sea level  
        dem = ee.Image('USGS/SRTMGL1_003')\
                .select('elevation')\
                .clip(ROI)\
                .reproject(crs=crs, scale=scale)\
                .updateMask(maize_mask)
        
        combined = ndvi_mean.addBands(ndvi_min)\
                            .addBands(ndvi_max)\
                            .addBands(pm25_mean)\
                            .addBands(pm25_max)\
                            .addBands(mean_temp)\
                            .addBands(max_temp)\
                            .addBands(prep)\
                            .addBands(mean_rh)\
                            .addBands(bm)\
                            .addBands(dem)
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=100,
                                    projection='EPSG:4326')


        def extract_properties(feature):
            properties = {
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1),
                'CropType': crop_type,
                'NDVI_mean ': feature.get('NDVI_mean'),
                'NDVI_min ': feature.get('NDVI_min'),
                'NDVI_max ': feature.get('NDVI_max'),
                'PM25_mean': feature.get('PM25_mean'),
                'PM25_max': feature.get('PM25_max'),
                'Temperature_mean': feature.get('Temperature_mean'),
                'Temperature_max': feature.get('Temperature_max'),
                'Rel_humidity_mean': feature.get('Rel_humidity_mean'),
                'Precipitation_sum': feature.get('Precipitation_sum'),
                'Luminosity': feature.get('Luminosity'),
                'elevation': feature.get('elevation')
            }
            return ee.Feature(None, properties)

        processed_fc = feat_coll.map(extract_properties)
        
        # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
        #         'NDVI_mean', 'PM25_mean']
        
        # task = ee.batch.Export.table.toDrive(collection=processed_fc,
        #                                     description=f'{tile}_{scale}m_{START_DATE}',
        #                                     fileFormat='CSV',
        #                                     selectors=bands,
        #                                     folder='wheat_south')

        # task.start()

        df = geemap.ee_to_df(processed_fc)

        df.to_csv(f'D:\Work\ADB\SAR\wheat_south\{tile}_{scale}m_{START_DATE}.csv', index=False)



<string>:181: SyntaxWarning: invalid escape sequence '\{'
<>:181: SyntaxWarning: invalid escape sequence '\{'
<>:181: SyntaxWarning: invalid escape sequence '\W'
<string>:181: SyntaxWarning: invalid escape sequence '\{'
<>:181: SyntaxWarning: invalid escape sequence '\{'
<>:181: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_4912\3119219348.py:181: SyntaxWarning: invalid escape sequence '\{'
  df.to_csv(f'D:\Work\ADB\SAR\wheat_south\{tile}_{scale}m_{START_DATE}.csv', index=False)
C:\Users\sonle\AppData\Local\Temp\ipykernel_4912\3119219348.py:181: SyntaxWarning: invalid escape sequence '\W'
  df.to_csv(f'D:\Work\ADB\SAR\wheat_south\{tile}_{scale}m_{START_DATE}.csv', index=False)


CRS: EPSG:4326
scale: 100
Start: 2017-01. End: 2022-12
  N09E105_1_0
    2017-01


In [9]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

def mask_crop_map(image):
    return image.updateMask(wheat_mask)

for tile in chosen_tiles[:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

        pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                        .mean()\
                        .multiply(1E9)\
                        .rename('PM25_mean')\
                        .clip(ROI)\
                        .reproject(crs='EPSG:4326', scale=100)
        
        pm25_min = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .min()\
                    .multiply(1E9)\
                    .rename('PM25_min')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
        
        pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
        
        
        combined = pm25_mean.addBands(pm25_max)\
                            .addBands(pm25_min)
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=100,
                                    projection='EPSG:4326')


        def extract_properties(feature):
            properties = {
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1),
                'CropType': crop_type,
                'PM25_mean': feature.get('PM25_mean'),
                'PM25_max': feature.get('PM25_max'),
                'PM25_min': feature.get('PM25_min')
            }
            return ee.Feature(None, properties)

        processed_fc = feat_coll.map(extract_properties)
        
        bands = ['Timestamp', 'lon', 'lat', 
                'PM25_mean', 'PM25_max', 'PM25_min']
        
        task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                            description=f'{tile}_{scale}m_{START_DATE}_pm25',
                                            fileFormat='CSV',
                                            selectors=bands,
                                            folder='wheat_south')

        task.start()

        # df = geemap.ee_to_df(processed_fc)

        # df.to_csv(f'D:\Work\ADB\SAR\wheat_south\{tile}_{scale}m_{START_DATE}_pm25.csv', index=False)



CRS: EPSG:4326
scale: 100
Start: 2017-01. End: 2022-12
  N10E105_0_1
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
    2018-01
    2018-02
    2018-03
    2018-04
    2018-05
    2018-06
    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2019-05
    2019-06
    2019-07
    2019-08
    2019-09
    2019-10
    2019-11
    2019-12
    2020-01
    2020-02
    2020-03
    2020-04
    2020-05
    2020-06
    2020-07
    2020-08
    2020-09
    2020-10
    2020-11
    2020-12
    2021-01
    2021-02
    2021-03
    2021-04
    2021-05
    2021-06
    2021-07
    2021-08
    2021-09
    2021-10
    2021-11
    2021-12
    2022-01
    2022-02
    2022-03
    2022-04
    2022-05
    2022-06
    2022-07
    2022-08
    2022-09
    2022-10
    2022-11
    2022-12
  N10E105_1_1
    2017-01
    2017-02
    2017-03
    2017-04
    2

In [6]:
# Load the WorldCereal dataset
dataset = ee.ImageCollection('ESA/WorldCereal/2021/MODELS/v100')

# Function to mask out "other" class
def mask_other(img):
    return img.updateMask(img.neq(0))

# Apply the mask_other function to the collection
dataset = dataset.map(mask_other)

# Filter for season and AEZ, and then filter on product
tc = dataset.filter(ee.Filter.eq('season', 'tc-wintercereals')) \
            .filter(ee.Filter.eq('aez_id', 18085)) \
            .filter(ee.Filter.eq('product', 'wintercereals'))

# Function to mask low classification and confidence values
def mask_low_classification_confidence(img):
    classification_mask = img.select('classification').gt(50)
    confidence_mask = img.select('confidence').gt(50)
    return img.updateMask(classification_mask).updateMask(confidence_mask)

# Apply the mask function to the image collection
tc = tc.map(mask_low_classification_confidence)

# Combine the masks from all images in the collection by using the mosaic() or max() to get a single mask
# Here, we use mosaic to combine all masks into one composite mask
tc_mask = tc.max().select('classification')

# Load the Sentinel-2 Harmonized dataset
s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')

# Apply the tc mask to the Sentinel-2 images
def apply_tc_mask(img):
    return img.updateMask(tc_mask)

# Apply the mask to the entire Sentinel-2 collection
s2_masked = s2.map(apply_tc_mask)

# Visualization parameters for Sentinel-2
visualization = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue bands
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

Map = geemap.Map()
# Display masked Sentinel-2 images
Map.addLayer(s2_masked.mean(), visualization, 'Masked Sentinel-2 Image')

# Center the map around the region of interest
Map.centerObject(tc, 6)
Map

Map(center=[10.808789144227068, 105.50266753796576], controls=(WidgetControl(options=['position', 'transparent…

## SAR

In [9]:
def calculate_rvi(image):
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI')

  return image.addBands(rvi, None, True)

def calculate_vvvh(image):
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv

crs = 'EPSG:4326'
scale=100



START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'SINGLE'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 2 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0
CLIP_TO_ROI = True

with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
# chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
#                 'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']    


<>:44: SyntaxWarning: invalid escape sequence '\W'
<>:44: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_28092\1601848277.py:44: SyntaxWarning: invalid escape sequence '\W'
  with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:


In [15]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

crop_type = 'Wheat'

def mask_crop_map(image):
    return image.updateMask(wheat_mask)

for tile in chosen_tiles[:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])

        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI
        s1_1 = s1_1.map(calculate_rvi)\
                   .map(mask_crop_map)
        


        rvi_mean = s1_1.mean()\
                .select("RVI")\
                .rename("RVI_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
                
        rvi_min = s1_1.min()\
                .select("RVI")\
                .rename("RVI_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
                
        rvi_max = s1_1.max()\
                .select("RVI")\
                .rename("RVI_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        
        vv_mean = s1_1.mean()\
                .select("VV")\
                .rename("VV_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
        
        vv_min = s1_1.min()\
                .select("VV")\
                .rename("VV_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vv_max = s1_1.max()\
                .select("VV")\
                .rename("VV_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vh_mean = s1_1.mean()\
                .select("VH")\
                .rename("VH_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
        
        vh_min = s1_1.min()\
                .select("VH")\
                .rename("VH_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vh_max = s1_1.max()\
                .select("VH")\
                .rename("VH_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vvvh_mean = s1_1.map(calculate_vvvh)\
                        .mean()\
                        .select("VVVH")\
                        .rename("VVVH_mean")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        
        vvvh_min = s1_1.map(calculate_vvvh)\
                        .min()\
                        .select("VVVH")\
                        .rename("VVVH_min")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
        
        vvvh_max = s1_1.map(calculate_vvvh)\
                        .max()\
                        .select("VVVH")\
                        .rename("VVVH_max")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        vhvv_mean = s1_1.map(calculate_vhvv)\
                        .mean()\
                        .select("VHVV")\
                        .rename("VHVV_mean")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        
        vhvv_min = s1_1.map(calculate_vhvv)\
                        .min()\
                        .select("VHVV")\
                        .rename("VHVV_min")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
        
        vhvv_max = s1_1.map(calculate_vhvv)\
                        .max()\
                        .select("VHVV")\
                        .rename("VHVV_max")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                
        
        
        combined = rvi_mean.addBands(rvi_min)\
                            .addBands(rvi_max)\
                            .addBands(vv_mean)\
                            .addBands(vv_min)\
                            .addBands(vv_max)\
                            .addBands(vh_mean)\
                            .addBands(vh_min)\
                            .addBands(vh_max)\
                            .addBands(vvvh_mean)\
                            .addBands(vvvh_min)\
                            .addBands(vvvh_max)\
                            .addBands(vhvv_mean)\
                            .addBands(vhvv_min)\
                            .addBands(vhvv_max)\
                            
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=scale,
                                    projection=crs)


        def add_properties(feature):
            
            return feature.set({
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1)
            })
        
        processed_fc = feat_coll.map(add_properties)
        
        # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'RVI_mean',
        #         'RVI_min', 'RVI_max']
        
        # task = ee.batch.Export.table.toDrive(collection=processed_fc,
        #                                     description=f'{tile}_{scale}m_{START_DATE}_wheat',
        #                                     fileFormat='CSV',
        #                                     selectors=['Timestamp', 'lon', 'lat', 'RVI_mean'],
        #                                     folder='wheat_south')

        # task.start()
        try:
            df = geemap.ee_to_df(processed_fc)

            df.to_csv(f'D:\Work\ADB\SAR\wheat_sar\{tile}_{scale}m_{START_DATE}.csv', index=False)
        except Exception as e:
            print('     failed')
            continue

<string>:209: SyntaxWarning: invalid escape sequence '\{'
<>:209: SyntaxWarning: invalid escape sequence '\{'
<>:209: SyntaxWarning: invalid escape sequence '\W'
<string>:209: SyntaxWarning: invalid escape sequence '\{'
<>:209: SyntaxWarning: invalid escape sequence '\{'
<>:209: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_28092\1353330091.py:209: SyntaxWarning: invalid escape sequence '\{'
  df.to_csv(f'D:\Work\ADB\SAR\wheat_sar\{tile}_{scale}m_{START_DATE}.csv', index=False)
C:\Users\sonle\AppData\Local\Temp\ipykernel_28092\1353330091.py:209: SyntaxWarning: invalid escape sequence '\W'
  df.to_csv(f'D:\Work\ADB\SAR\wheat_sar\{tile}_{scale}m_{START_DATE}.csv', index=False)


CRS: EPSG:4326
scale: 100
Start: 2017-01. End: 2022-12
  N10E105_0_1
    2017-01
    2017-02
    2017-03
    2017-04
    2017-05
    2017-06
    2017-07
    2017-08
    2017-09
    2017-10
    2017-11
    2017-12
    2018-01
    2018-02
    2018-03
    2018-04
    2018-05
    2018-06
    2018-07
    2018-08
    2018-09
    2018-10
    2018-11
    2018-12
    2019-01
    2019-02
    2019-03
    2019-04
    2019-05
    2019-06
    2019-07
    2019-08
    2019-09
    2019-10
    2019-11
    2019-12
    2020-01
    2020-02
    2020-03
    2020-04
    2020-05
    2020-06
    2020-07
    2020-08
    2020-09
    2020-10
    2020-11
    2020-12
    2021-01
    2021-02
    2021-03
    2021-04
    2021-05
    2021-06
    2021-07
    2021-08
    2021-09
    2021-10
    2021-11
    2021-12
    2022-01
    2022-02
    2022-03
    2022-04
    2022-05
    2022-06
    2022-07
    2022-08
    2022-09
    2022-10
    2022-11
    2022-12
  N10E105_1_1
    2017-01
    2017-02
    2017-03
    2017-04
    2

## Provincial GSO

In [95]:
poly = gpd.read_file('shapefiles/gadm41_VNM_1.json')
poly.head()
# r = poly.iloc[:1]
# regions = geemap.gdf_to_ee(r).geometry()

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,VNM.1_1,VNM,Vietnam,AnGiang,AnGiang,NA,Tỉnh,Province,NA,VN.AG,VN-44,"MULTIPOLYGON (((105.54860 10.42950, 105.54950 ..."
1,VNM.7_1,VNM,Vietnam,BàRịa-VũngTàu,BaRia-VungTau,NA,Tỉnh,Province,NA,VN.BV,NA,"MULTIPOLYGON (((107.09010 10.32400, 107.08890 ..."
2,VNM.3_1,VNM,Vietnam,BắcGiang,BacGiang,NA,Tỉnh,Province,NA,VN.BG,NA,"MULTIPOLYGON (((106.28380 21.13230, 106.27340 ..."
3,VNM.4_1,VNM,Vietnam,BắcKạn,BacKan,NA,Tỉnh,Province,NA,VN.BK,NA,"MULTIPOLYGON (((105.87240 21.85580, 105.86290 ..."
4,VNM.2_1,VNM,Vietnam,BạcLiêu,BacLieu,NA,Tỉnh,Province,NA,VN.BL,NA,"MULTIPOLYGON (((105.42440 9.02130, 105.41640 9..."


In [96]:
names = poly['GID_1'].to_list()
names.sort()

In [11]:
dates1 = ['2015-01-01', '2015-06-30', '2016-01-01', '2016-06-30', '2017-01-01', '2017-06-30', '2018-01-01', '2018-06-30', '2019-01-01', '2019-06-30', '2020-01-01', '2020-06-30', '2021-01-01', '2021-06-30', '2022-01-01', '2022-06-30']
dates2 = ['2015-06-30', '2016-01-01', '2016-06-30', '2017-01-01', '2017-06-30', '2018-01-01', '2018-06-30', '2019-01-01', '2019-06-30', '2020-01-01', '2020-06-30', '2021-01-01', '2021-06-30', '2022-01-01', '2022-06-30', '2023-01-01']


# dates1 = ['2021-01-01', '2021-03-31', '2021-06-30', '2021-09-30', '2021-12-31', '2022-03-31', '2022-06-30', '2022-09-30']
# dates2 = ['2021-03-31', '2021-06-30', '2021-09-30', '2021-12-31', '2022-03-31', '2022-06-30', '2022-09-30', '2022-12-31']

In [12]:
with open('JAXA_subtiles_full.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)
jaxa_subtiles

FileNotFoundError: [Errno 2] No such file or directory: 'JAXA_subtiles_full.json'

In [71]:
APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'MULTI'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 3 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0


In [38]:

# regions = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.inList('ADM0_NAME', ['Viet Nam'])).geometry()

# JAXA LULC VN dataset
# https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
# jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN')
# mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class
# START_YM = '2022-01'
# END_YM = '2022-12'
# months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')   
              
for d in range(len(dates1))[:1]:
    START_DATE = dates1[d]
    STOP_DATE = dates2[d]
    print(dates1[d])
# for month in months[:4]:
# for year in list(range(2017, 2023, 1)):
    # print(f"    {month}")

    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
    for tile in list(jaxa_subtiles.keys()):
      print(f"  {tile}")
      bbox = jaxa_subtiles[tile]
      regions = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])
    # for i in range(len(poly)):
      
      # r = poly.iloc[i:i+1]
      # name = names[i]
      # print(f'  {name}')
      # regions = geemap.gdf_to_ee(r).geometry()

      # jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      # mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])

        # Border noise correction
      # s1_1 = s1.map(bnc.f_mask_edges)

        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI_1

      # rvi = s1_1.map(calculate_rvi)\
      #           .mean()
      s1_1 = s1_1.mean()\
                .select(['VV', 'VH'])
                
      
      task = ee.batch.Export.image.toDrive(image=s1_1,
                                      description=f'{tile}_{dates1[d]}',
                                      scale=100,
                                      region=regions,
                                      folder='images',
                                      fileNamePrefix=f'{tile}_{dates1[d]}',
                                      crs='EPSG:4326',
                                      fileFormat='GeoTIFF')
      task.start()
    


2021-01-01
  N08E104_0_0
  N08E104_0_1
  N08E104_1_0
  N08E104_1_1
  N08E105_0_0
  N08E105_0_1
  N08E105_1_0
  N08E105_1_1
  N09E104_0_0
  N09E104_0_1
  N09E104_1_0
  N09E104_1_1
  N09E105_0_0
  N09E105_0_1
  N09E105_1_0
  N09E105_1_1
  N09E106_0_0
  N09E106_0_1
  N09E106_1_0
  N09E106_1_1
  N10E103_0_0
  N10E103_0_1
  N10E103_1_0
  N10E103_1_1
  N10E104_0_0
  N10E104_0_1
  N10E104_1_0
  N10E104_1_1
  N10E105_0_0
  N10E105_0_1
  N10E105_1_0
  N10E105_1_1
  N10E106_0_0
  N10E106_0_1
  N10E106_1_0
  N10E106_1_1
  N10E107_0_0
  N10E107_0_1
  N10E107_1_0
  N10E107_1_1
  N10E108_0_0
  N10E108_0_1
  N10E108_1_0
  N10E108_1_1
  N11E105_0_0
  N11E105_0_1
  N11E105_1_0
  N11E105_1_1
  N11E106_0_0
  N11E106_0_1
  N11E106_1_0
  N11E106_1_1
  N11E107_0_0
  N11E107_0_1
  N11E107_1_0
  N11E107_1_1
  N11E108_0_0
  N11E108_0_1
  N11E108_1_0
  N11E108_1_1
  N11E109_0_0
  N11E109_0_1
  N11E109_1_0
  N11E109_1_1
  N12E106_0_0
  N12E106_0_1
  N12E106_1_0
  N12E106_1_1
  N12E107_0_0
  N12E107_0_1
  N12E107

KeyboardInterrupt: 

In [ ]:
def calculate_dprvi(image):
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb
  q = image.select('VH').divide(image.select('VV'))
  q = q.where(q.lt(0), 0)
  q = q.where(q.gt(1), 1)
  N = q.multiply(q.add(3))
  D = (q.add(1)).multiply(q.add(1))
  dprvi = N.divide(D).rename('DPRVI_mean')

  return dprvi

def calculate_rvi(image):
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI_mean')

  return rvi

def calculate_vvvh(image):
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv



In [8]:
seasons = {'early_spring': ['{}-01-15', '{}-02-15'],
           'late_spring': ['{}-05-01', '{}-05-31'],
           'main_autumn': ['{}-06-01', '{}-07-31'],
           'late_autumn': ['{}-06-15', '{}-12-15'],
           'winter': ['{}-11-01', '{}-11-30'],
           'rice_1_s': ['{}-01-01', '{}-01-31'],
           'rice_2_s': ['{}-07-01', '{}-07-31'],
           'rice_3_s': ['{}-09-01', '{}-09-31'],
           'rice_1_n': ['{}-09-01', '{}-09-31'],
           'rice_2_n': ['{}-07-01', '{}-07-31'],
           'rice_3_n': ['{}-10-01', '{}-10-30']}

### SAR

In [11]:
season = 'early_spring'
print(season)
# for d in range(len(dates1))[4:5]:
#     START_DATE = dates1[d]
#     STOP_DATE = dates2[d]
#     print(dates1[d])
# for month in months[:4]:
    # print(f"    {month}")
    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
for year in list(range(2017, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # for tile in list(jaxa_subtiles.keys())[:50]:
    #   print(f"  {tile}")
    #   bbox = jaxa_subtiles[tile]
    #   regions = ee.Geometry.BBox(bbox[0],
    #                         bbox[1],
    #                         bbox[2],
    #                         bbox[3])
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))


      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

early_spring
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2018
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.

In [12]:
season = 'late_spring'
print(season)
# for d in range(len(dates1))[4:5]:
#     START_DATE = dates1[d]
#     STOP_DATE = dates2[d]
#     print(dates1[d])
# for month in months[:4]:
    # print(f"    {month}")
    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
for year in list(range(2017, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # for tile in list(jaxa_subtiles.keys())[:50]:
    #   print(f"  {tile}")
    #   bbox = jaxa_subtiles[tile]
    #   regions = ee.Geometry.BBox(bbox[0],
    #                         bbox[1],
    #                         bbox[2],
    #                         bbox[3])
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI_1

      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

late_spring
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2018
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.2

In [13]:
season = 'winter'
print(season)
# for d in range(len(dates1))[4:5]:
#     START_DATE = dates1[d]
#     STOP_DATE = dates2[d]
#     print(dates1[d])
# for month in months[:4]:
    # print(f"    {month}")
    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
for year in list(range(2017, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # for tile in list(jaxa_subtiles.keys())[:50]:
    #   print(f"  {tile}")
    #   bbox = jaxa_subtiles[tile]
    #   regions = ee.Geometry.BBox(bbox[0],
    #                         bbox[1],
    #                         bbox[2],
    #                         bbox[3])
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI_1

      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

winter
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2018
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
 

In [14]:
season = 'main_autumn'
print(season)
# for d in range(len(dates1))[4:5]:
#     START_DATE = dates1[d]
#     STOP_DATE = dates2[d]
#     print(dates1[d])
# for month in months[:4]:
    # print(f"    {month}")
    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
for year in list(range(2017, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # for tile in list(jaxa_subtiles.keys())[:50]:
    #   print(f"  {tile}")
    #   bbox = jaxa_subtiles[tile]
    #   regions = ee.Geometry.BBox(bbox[0],
    #                         bbox[1],
    #                         bbox[2],
    #                         bbox[3])
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI_1

      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

main_autumn
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2018
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.2

In [15]:
season = 'late_autumn'
print(season)
# for d in range(len(dates1))[4:5]:
#     START_DATE = dates1[d]
#     STOP_DATE = dates2[d]
#     print(dates1[d])
# for month in months[:4]:
    # print(f"    {month}")
    # START_DATE = month + '-01'
    # STOP_DATE = get_last_day_of_month_from_Ym(month,
    #                                         input_format='%Y-%m',
    #                                         output_format='%Y-%m-%d')
for year in list(range(2017, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # for tile in list(jaxa_subtiles.keys())[:50]:
    #   print(f"  {tile}")
    #   bbox = jaxa_subtiles[tile]
    #   regions = ee.Geometry.BBox(bbox[0],
    #                         bbox[1],
    #                         bbox[2],
    #                         bbox[3])
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI_1

      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

late_autumn
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2018
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.2

In [13]:
poly[poly['GID_1'] == 'VNM.26_1']

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
21,VNM.26_1,VNM,Vietnam,HàGiang,HaGiang,NA,Tỉnh,Province,NA,VN.HG,NA,"MULTIPOLYGON (((104.72980 22.22550, 104.72840 ..."


In [ ]:
# display(stats)

In [158]:
stats

In [161]:
type(stats.get('DPRVI_mean'))

ee.computedobject.ComputedObject

In [162]:
stats.getInfo()

{'DPRVI_mean': 0.4295412056688735,
 'RVI_mean': 0.6453070529356154,
 'VH': 0.05035580325004624,
 'VHVV_mean': 0.19438613456617584,
 'VV': 0.2708362503873179,
 'VVVH_mean': 5.47172525667059,
 'angle': 36.86089824193414}

In [156]:
processed_stats.get('RVI_mean')

### NDVI/LAI


In [99]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("EVI")

    image = image.addBands(EVI)

    return(image)

def getLAI(image):
    LAI = image.expression(
        '(3.618*EVI - 0.118)', {
            'EVI': image.select('EVI')
        }).rename("LAI_mean")
    image = image.addBands(LAI)

    return(image)

def getNDVI(image):
    
    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI_mean")
    image = image.addBands(ndvi)

    return(image)

def mask_crop_map(image):
    return image.updateMask(mask_jaxa)


In [100]:
main_dir = os.getcwd().replace('\\', '/')
main_dir

'd:/Work/ADB/SAR'

In [24]:
season = 'winter'
print(season)

for year in list(range(2016, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

      jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
      mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      


      # # RVI
      # s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
      #         .filter(ee.Filter.eq('instrumentMode', 'IW'))\
      #         .filter(ee.Filter.eq('resolution_meters', 10)) \
      #         .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
      #         .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
      #         .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
      #         .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
      #         .filterDate(START_DATE, STOP_DATE)\
      #         .select(['VV', 'VH', 'angle'])
      
      # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
      # Sentinel-2 1C -> radiometrically calibrated
      s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)


      #   # Speckle filter
      # s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
      #                                                 SPECKLE_FILTER_KERNEL_SIZE,
      #                                                 SPECKLE_FILTER))

      # # Terrain correction
      # s1_1 = (trf.slope_correction(s1_1,
      #                             TERRAIN_FLATTENING_MODEL,
      #                             DEM,
      #                             TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
      lai = s2.map(getEVI)\
              .map(getLAI)\
              .mean()\
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
      nvdi = s2.map(getNDVI)\
              .mean()

      combined = nvdi.addBands(lai)


      # dprvi = s1_1.map(calculate_dprvi)\
      #           .mean()\
      #           .updateMask(mask_jaxa)

      # rvi = s1_1.map(calculate_rvi)\
      #           .mean()\
      #           .updateMask(mask_jaxa)
      
      # vvvh = s1_1.map(calculate_vvvh)\
      #           .mean()\
      #           .updateMask(mask_jaxa)
      

      # vhvv = s1_1.map(calculate_vhvv)\
      #           .mean()\
      #           .updateMask(mask_jaxa)
      
      # img = s1_1.mean()\
      #           .updateMask(mask_jaxa)
      
      # combined = img.addBands(dprvi)\
      #               .addBands(rvi)\
      #               .addBands(vvvh)\
      #               .addBands(vhvv)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal_s2/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

winter
  2016
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
 

'd:\\Work\\ADB\\SAR'

### Combined

In [103]:
season = 'early_spring'
print(season)

for year in list(range(2016, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      regions = geemap.gdf_to_ee(r).geometry()

    #   jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
    #   mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class


      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])
      
      # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
      # Sentinel-2 1C -> radiometrically calibrated
      s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            # .map(mask_crop_map)


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
      lai = s2.map(getEVI)\
              .map(getLAI)\
              .mean()\
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
      nvdi = s2.map(getNDVI)\
              .mean()




      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                # .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                # .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                # .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                # .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                # .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)\
                    .addBands(nvdi)\
                    .addBands(lai)
      
    #   combined = nvdi.addBands(lai)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        # test_d['date'] = dates1[d]
        test_d['date'] = START_DATE
        test_d['season'] = season
        # dict_list.append(test_d)
        with open(f'datasets/seasonal_no_mask/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

early_spring
  2016
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.21_1
    VNM.22_1
    VNM.23_1
    VNM.24_1
    VNM.25_1
    VNM.26_1
    VNM.27_1
    VNM.28_1
    VNM.29_1
    VNM.2_1
    VNM.30_1
    VNM.31_1
    VNM.32_1
    VNM.33_1
    VNM.34_1
    VNM.35_1
    VNM.36_1
    VNM.37_1
    VNM.38_1
    VNM.39_1
    VNM.3_1
    VNM.40_1
    VNM.41_1
    VNM.42_1
    VNM.43_1
    VNM.44_1
    VNM.45_1
    VNM.46_1
    VNM.47_1
    VNM.48_1
    VNM.49_1
    VNM.4_1
    VNM.50_1
    VNM.51_1
    VNM.52_1
    VNM.53_1
    VNM.54_1
    VNM.55_1
    VNM.56_1
    VNM.57_1
    VNM.58_1
    VNM.59_1
    VNM.5_1
    VNM.60_1
    VNM.61_1
    VNM.62_1
    VNM.63_1
    VNM.6_1
    VNM.7_1
    VNM.8_1
    VNM.9_1
  2017
    VNM.10_1
    VNM.11_1
    VNM.12_1
    VNM.13_1
    VNM.14_1
    VNM.15_1
    VNM.16_1
    VNM.17_1
    VNM.18_1
    VNM.19_1
    VNM.1_1
    VNM.20_1
    VNM.

# In-situ Data

In [101]:
import datetime
import geemap
from tqdm import tqdm_notebook

In [102]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("EVI")

    image = image.addBands(EVI)

    return(image)

def getLAI(image):
    LAI = image.expression(
        '(3.618*EVI - 0.118)', {
            'EVI': image.select('EVI')
        }).rename("LAI")
    image = image.addBands(LAI)

    return(image)

def getNDVI(image):
    
    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI")
    image = image.addBands(ndvi)

    return(image)

def mask_crop_map(image):
    return image.updateMask(mask_jaxa)

def calculate_dprvi(image):
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb
  q = image.select('VH').divide(image.select('VV'))
  q = q.where(q.lt(0), 0)
  q = q.where(q.gt(1), 1)
  N = q.multiply(q.add(3))
  D = (q.add(1)).multiply(q.add(1))
  dprvi = N.divide(D).rename('DPRVI')

  return dprvi

def calculate_rvi(image):
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI')

  return rvi

def calculate_vvvh(image):
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv



In [103]:
APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'MULTI'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 3 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0


## EY Dataset

In [15]:
dataset = pd.read_csv('datasets/Crop_Yield_Data_challenge_2.csv')
dataset['field_size_m2'] = dataset['Field size (ha)'] * 10000
dataset['radius'] = np.sqrt(dataset['field_size_m2'] / np.pi)
dataset['side'] = np.sqrt(dataset['field_size_m2'])
dataset['id'] = dataset.index

lats = dataset['Latitude'].to_list()
lons = dataset['Longitude'].to_list()
dates = dataset['Date of Harvest'].to_list()
radii = dataset['radius'].to_list()
sides = dataset['side'].to_list()

dataset.head()

,District,Latitude,Longitude,"Season(SA = Summer Autumn, WS = Winter Spring)","Rice Crop Intensity(D=Double, T=Triple)",Date of Harvest,Field size (ha),Rice Yield (kg/ha),field_size_m2,radius,side,id
0,Chau_Phu,10.510542,105.248554,SA,T,15-07-2022,3.40,5500,34000.0,104.031419,184.390889,0
1,Chau_Phu,10.509150,105.265098,SA,T,15-07-2022,2.43,6000,24300.0,87.948452,155.884573,1
2,Chau_Phu,10.467721,105.192464,SA,D,15-07-2022,1.95,6400,19500.0,78.784788,139.642400,2
3,Chau_Phu,10.494453,105.241281,SA,T,15-07-2022,4.30,6000,43000.0,116.992842,207.364414,3
4,Chau_Phu,10.535058,105.252744,SA,D,14-07-2022,3.30,6400,33000.0,102.490128,181.659021,4


In [16]:
from shapely import wkt

def create_poly(lat, lon, distance, type='round'):
    gs = gpd.GeoSeries(wkt.loads(f'POINT ({lon} {lat})'))
    gdf = gpd.GeoDataFrame(geometry=gs)
    gdf.crs='EPSG:4326'
    gdf = gdf.to_crs('EPSG:3857')
    res = gdf.buffer(
        distance=distance,
        cap_style=type,
    )

    return res.to_crs('EPSG:4326').iloc[0]

circles = []
squares = []

for i in tqdm_notebook(range(len(lats))):
    circles.append(create_poly(lats[i], lons[i], sides[i], type='round'))
    squares.append(create_poly(lats[i], lons[i], sides[i], type='square'))

gdf_circle = gpd.GeoDataFrame(dataset, crs='EPSG:4326', geometry=circles)
gdf_square = gpd.GeoDataFrame(dataset, crs='EPSG:4326', geometry=squares)
    

C:\Users\sonle\AppData\Local\Temp\ipykernel_36388\1232142023.py:18: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm_notebook(range(len(lats))):


  0%|          | 0/557 [00:00<?, ?it/s]

In [20]:
for i in range(len(gdf_square))[:]:
      
    gdf = gdf_circle.iloc[i:i+1].reset_index(drop=True)
    regions = geemap.gdf_to_ee(gdf).geometry()

    jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
    mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

    name = gdf['id'][0]
    date = gdf['Date of Harvest'][0]
    print(name)

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(28)).strftime('%Y-%m-%d')
    STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


    # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

    # RVI

    dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

    rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

    vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vv_n_vh = s1_1.mean()\
                .updateMask(mask_jaxa)


    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)

      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
    lai = s2.map(getEVI)\
            .map(getLAI)\
            .mean()\
            .select('LAI')
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
    ndvi = s2.map(getNDVI)\
             .mean()\
             .select('NDVI')

      
    combined = vv_n_vh.addBands(dprvi)\
                      .addBands(rvi)\
                      .addBands(vvvh)\
                      .addBands(vhvv)\
                      .addBands(lai)\
                      .addBands(ndvi)

    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.max(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.min(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=reducers,
                      scale=10,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['id'] = str(name)
        test_d['date'] = date
        with open(f'datasets/EY/square_4w/{name}.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [88]:
for i in range(len(gdf_square)):
      
    gdf = gdf_square.iloc[i:i+1].reset_index(drop=True)
    regions = geemap.gdf_to_ee(gdf).geometry()

    jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
    mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

    name = gdf['id'][0]
    date = gdf['Date of Harvest'][0]
    print(date)

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(7)).strftime('%Y-%m-%d')
    STOP_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') + datetime.timedelta(7)).strftime('%Y-%m-%d')

    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


    # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

    # RVI

    dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

    rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

    vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vv_n_vh = s1_1.mean()\
                .updateMask(mask_jaxa)


    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)

      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
    lai = s2.map(getEVI)\
            .map(getLAI)\
            .mean()\
            .select('LAI_mean')
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
    ndvi = s2.map(getNDVI)\
             .mean()\
             .select('NDVI_mean')

      
    combined = vv_n_vh.addBands(dprvi)\
                      .addBands(rvi)\
                      .addBands(vvvh)\
                      .addBands(vhvv)\
                      .addBands(lai)\
                      .addBands(ndvi)


    stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['id'] = str(name)
        test_d['date'] = date
        with open(f'datasets/EY/square_mask/{name}.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue

15-07-2022
15-07-2022
15-07-2022
15-07-2022
14-07-2022
15-07-2022
15-07-2022
15-07-2022
15-07-2022
15-07-2022
15-07-2022
15-07-2022
13-07-2022
13-07-2022
14-07-2022
15-07-2022
14-07-2022
14-07-2022
14-07-2022
14-07-2022
14-07-2022
20-07-2022
20-07-2022
20-07-2022
20-07-2022
20-07-2022
20-07-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
09-08-2022
09-08-2022
09-08-2022
09-08-2022
09-08-2022
09-08-2022
09-08-2022
09-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
28-07-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
05-08-2022
01-04-2022
01-04-2022
01-04-2022
02-04-2022
01-04-2022
01-04-2022
01-04-2022
03-04-2022
04-04-2022
01-04-2022
01-04-2022
12-04-2022
12-04-2022
06-04-2022
06-04-2022
10-04-2022
10-04-2022
10-04-2022

## ADB

In [122]:
import math
from shapely import Polygon

def meters_to_degrees_wgs84(latitude, meters):
    # 1 degree latitude ≈ 111.32 km or 111320 meters
    lat_degree = meters / 111320
    
    # 1 degree longitude ≈ 111.32 km * cos(latitude), where latitude is in radians
    lon_degree = meters / (111320 * math.cos(math.radians(latitude)))
    
    return lat_degree, lon_degree

def create_bounding_box(lat, lon, side_length_m=2.5):
    # Convert side length in meters to degrees
    lat_delta, lon_delta = meters_to_degrees_wgs84(lat, side_length_m)
    
    # Calculate min and max for latitude and longitude (bounding box)
    min_lat = lat
    max_lat = lat + lat_delta
    min_lon = lon
    max_lon = lon + lon_delta
    
    return [min_lon, min_lat, max_lon, max_lat]

def create_geodataframe_from_bbox(bbox):
    # Create a polygon from the bounding box
    min_lon, min_lat, max_lon, max_lat = bbox
    polygon = Polygon([
        (min_lon, min_lat),  # Bottom-left
        (max_lon, min_lat),  # Bottom-right
        (max_lon, max_lat),  # Top-right
        (min_lon, max_lat),  # Top-left
        (min_lon, min_lat)   # Closing the polygon
    ])


def calculate_area_from_bbox(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    
    # Calculate the latitude and longitude differences in degrees
    delta_lat = max_lat - min_lat
    delta_lon = max_lon - min_lon
    
    # Convert the latitude difference to meters (latitude is constant)
    lat_distance_meters = delta_lat * 111320  # 1 degree latitude ≈ 111.32 km
    
    # Convert the longitude difference to meters (varies with latitude)
    # Use the average latitude for more accurate calculation
    avg_lat = (min_lat + max_lat) / 2
    lon_distance_meters = delta_lon * (111320 * math.cos(math.radians(avg_lat)))
    
    # Calculate the area in square meters
    area_m2 = lat_distance_meters * lon_distance_meters
    return area_m2


# Example usage with a point (20.621192 N, 106.379849 E)
latitude = 20.621192
longitude = 106.379849

bbox = create_bounding_box(latitude, longitude)
print(bbox)
print(calculate_area_from_bbox(bbox))


[106.379849, 20.621192, 106.37987299516556, 20.621214457779374]
6.2499995378527915


In [120]:
df = pd.read_csv(r'D:\Work\ADB\SAR\datasets\thai_binh_insitu.csv')
# df['longitude'] = df['longitude'].astype(float)

df['bounds'] = df.apply(lambda row: create_bounding_box(row['latitude'], row['longitude']), axis=1)
df['area'] = df.apply(lambda row: calculate_area_from_bbox(row['bounds']), axis=1)
df['harvest_date'] = df['harvest_date'].str.replace('/', '-')
df


,latitude,longitude,paddy_weight_after_threshing,paddy_weight_after_drying,harvest_date,bounds,area
0,20.621192,106.379849,3967.61,3001.61,5-10-2015,"[106.379849, 20.621192, 106.37987299516556, 20...",6.25
1,20.622097,106.379330,4461.21,3504.88,5-10-2015,"[106.37933, 20.622097, 106.37935399530818, 20....",6.25
2,20.621892,106.380214,4361.34,3376.35,5-10-2015,"[106.380214, 20.621892, 106.38023799527588, 20...",6.25
3,20.622102,106.379718,4100.02,3084.21,5-10-2015,"[106.379718, 20.622102, 106.37974199530898, 20...",6.25
4,20.622102,106.379733,4162.12,3272.97,6-10-2015,"[106.379733, 20.622102, 106.37975699530898, 20...",6.25
...,...,...,...,...,...,...,...
251,20.620496,106.185483,4210.50,3334.00,25-9-2015,"[106.185483, 20.620496, 106.18550699505589, 20...",6.25
252,20.625563,106.158519,5513.20,4122.30,25-9-2015,"[106.158519, 20.625563, 106.15854299585449, 20...",6.25
253,20.625591,106.157891,4414.23,3268.96,25-9-2015,"[106.157891, 20.625591, 106.1579149958589, 20....",6.25
254,20.625854,106.158675,4010.40,2875.80,25-9-2015,"[106.158675, 20.625854, 106.15869899590037, 20...",6.25


In [123]:
bboxes = df['bounds'].to_list()
dates = df['harvest_date'].to_list()

In [133]:
for i in tqdm_notebook(range(len(bboxes))):
    bbox = bboxes[i]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])\
                    .bounds(proj='EPSG:4326')
    
    date = dates[i]

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(28)).strftime('%Y-%m-%d')
    STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


    # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

    # RVI

    dprvi = s1_1.map(calculate_dprvi)\
                .mean()\

    rvi = s1_1.map(calculate_rvi)\
                .mean()\
      
    vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
      

    vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
      
    vv_n_vh = s1_1.mean()\


    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
    lai = s2.map(getEVI)\
            .map(getLAI)\
            .mean()\
            .select('LAI')
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
    ndvi = s2.map(getNDVI)\
             .mean()\
             .select('NDVI')

      
    combined = vv_n_vh.addBands(dprvi)\
                      .addBands(rvi)\
                      .addBands(vvvh)\
                      .addBands(vhvv)\
                      .addBands(lai)\
                      .addBands(ndvi)

    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.max(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.min(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
                      geometry=ROI,
                      reducer=reducers,
                      scale=10,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['date'] = date
        with open(f'datasets/thai_binh/4w/{i}.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue
    

C:\Users\sonle\AppData\Local\Temp\ipykernel_10252\3924322647.py:1: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm_notebook(range(len(bboxes))):


  0%|          | 0/256 [00:00<?, ?it/s]

In [134]:
for w in [1, 2, 3, 5, 6]:
    print(w)
    for i in tqdm_notebook(range(len(bboxes))):
        bbox = bboxes[i]
        ROI = ee.Geometry.BBox(bbox[0],
                                bbox[1],
                                bbox[2],
                                bbox[3])\
                        .bounds(proj='EPSG:4326')
        
        date = dates[i]

        START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(7*w)).strftime('%Y-%m-%d')
        STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI

        dprvi = s1_1.map(calculate_dprvi)\
                    .mean()\

        rvi = s1_1.map(calculate_rvi)\
                    .mean()\
        
        vvvh = s1_1.map(calculate_vvvh)\
                    .mean()\
        

        vhvv = s1_1.map(calculate_vhvv)\
                    .mean()\
        
        vv_n_vh = s1_1.mean()\


        s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
        # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
        lai = s2.map(getEVI)\
                .map(getLAI)\
                .mean()\
                .select('LAI')
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi = s2.map(getNDVI)\
                .mean()\
                .select('NDVI')

        
        combined = vv_n_vh.addBands(dprvi)\
                        .addBands(rvi)\
                        .addBands(vvvh)\
                        .addBands(vhvv)\
                        .addBands(lai)\
                        .addBands(ndvi)

        reducers = ee.Reducer.mean().combine(
            reducer2=ee.Reducer.max(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.min(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        )

        stats = combined.reduceRegion(
                        geometry=ROI,
                        reducer=reducers,
                        scale=10,  
                        crs='EPSG:4326', 
                        tileScale=14,
                        bestEffort=True
                    )

        try:
            test_d = stats.getInfo()
            test_d['date'] = date
            with open(f'datasets/thai_binh/{w}w/{i}.json', 'w') as out_file:
                json.dump(test_d, out_file)
        except Exception as e:
            print(e)
            continue

1


C:\Users\sonle\AppData\Local\Temp\ipykernel_10252\2181210923.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm_notebook(range(len(bboxes))):


  0%|          | 0/256 [00:00<?, ?it/s]

2


  0%|          | 0/256 [00:00<?, ?it/s]

...

  0%|          | 0/256 [00:00<?, ?it/s]

5


  0%|          | 0/256 [00:00<?, ?it/s]

6


  0%|          | 0/256 [00:00<?, ?it/s]